# Year-over-Year Spend Growth Problem

## Problem Description
We need to calculate the **year-over-year (YoY) spend growth** for each product.  

- For each product, compute the **total spend per year**.  
- Compare each year’s spend with the **previous year’s spend**.  
- Calculate the YoY growth rate as:  
  

\[
  \text{YoY Growth} = \frac{\text{Current Year Spend} - \text{Previous Year Spend}}{\text{Previous Year Spend}} \times 100
  \]


- If there is no previous year’s spend, return `NULL` for both previous spend and growth rate.  
- Output should be ordered by `product_id` and year in ascending order.

---

## Schema

### Table: Transactions
| Column Name       | Type     | Description                          |
|-------------------|----------|--------------------------------------|
| transaction_id    | INT      | Unique transaction identifier        |
| product_id        | INT      | Product identifier                   |
| spend             | DECIMAL  | Amount spent in the transaction      |
| transaction_date  | DATETIME | Date and time of the transaction     |

---

## Sample Data

### Transactions
| transaction_id | product_id | spend   | transaction_date    |
|----------------|------------|---------|---------------------|
| 1              | 123424     | 1500.60 | 2019-07-01 10:00:00 |
| 2              | 123424     | 1000.20 | 2020-05-15 12:00:00 |
| 3              | 123424     | 1246.44 | 2021-08-20 09:30:00 |
| 4              | 123424     | 2145.32 | 2022-11-11 14:45:00 |

---

## Expected Output

| year | product_id | current_spend | prev_spend | yoy_growth |
|------|------------|---------------|------------|------------|
| 2019 | 123424     | 1500.60       | NULL       | NULL       |
| 2020 | 123424     | 1000.20       | 1500.60    | -33.35     |
| 2021 | 123424     | 1246.44       | 1000.20    | 24.62      |
| 2022 | 123424     | 2145.32       | 1246.44    | 72.12      |

---

## PySpark Code: Create DataFrame and Temp View

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DecimalType, TimestampType , FloatType
from datetime import datetime

# Schema for Transactions
transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("spend", FloatType(), False),
    StructField("transaction_date", TimestampType(), False)
])

# Data for Transactions
transactions_data = [
    (1, 123424, 1500.60, datetime(2019,7,1,10,0,0)),
    (2, 123424, 1000.20, datetime(2020,5,15,12,0,0)),
    (3, 123424, 1246.44, datetime(2021,8,20,9,30,0)),
    (4, 123424, 2145.32, datetime(2022,11,11,14,45,0))
]

# Create DataFrame
transactions_df = spark.createDataFrame(transactions_data, transactions_schema)

# Register Temp View
transactions_df.createOrReplaceTempView("Transactions")

# Quick check
transactions_df.show()


In [0]:
%sql

with	cte AS (
			SELECT year(transaction_date) AS year,
				product_id,
				cast(spend AS DECIMAL(10, 2)) AS current_spend,
				CASE 
					WHEN year(lag(transaction_date, 1) OVER (
								PARTITION BY product_id ORDER BY year(transaction_date) ASC
								)) = year(transaction_date) - 1
						THEN cast(lag(spend, 1) OVER (
									PARTITION BY product_id ORDER BY year(transaction_date) ASC
									) AS DECIMAL(10, 2))
					ELSE NULL
					END AS prev_spend
			FROM Transactions
			)

	SELECT year,
		product_id,
		current_spend,
		prev_spend,
		CASE 
			WHEN prev_spend IS NULL
				THEN NULL
			ELSE casT(100 * cast((current_spend - coalesce(prev_spend, 0)) AS DECIMAL(10, 2)) / cast(coalesce(prev_spend, 0) AS DECIMAL(10, 2)) AS DECIMAL(10, 2))
			END AS yoy_growth
	FROM cte



### Explanation
- **`YEAR(transaction_date)`** → Extracts the year from the transaction date.  
- **`CAST(spend AS DECIMAL(10,2))`** → Ensures spend values are stored with two decimal places (important for financial data).  
- **`LAG(transaction_date, 1)`** → Looks at the previous transaction date for the same product.  
- **`CASE` logic**:
  - If the previous transaction year is exactly one year before the current year, we take the previous spend (`prev_spend`).  
  - Otherwise, we set `prev_spend` to `NULL` (meaning no valid previous year to compare).  

This step prepares a table with:
- `year`
- `product_id`
- `current_spend`
- `prev_spend` (only if the previous year exists)

---

## Step 2: Calculate YoY Growth
```sql
SELECT 
    year,
    product_id,
    current_spend,
    prev_spend,
    CASE 
        WHEN prev_spend IS NULL
            THEN NULL
        ELSE CAST(
            100 * CAST((current_spend - COALESCE(prev_spend, 0)) AS DECIMAL(10, 2)) 
                / CAST(COALESCE(prev_spend, 0) AS DECIMAL(10, 2)) 
            AS DECIMAL(10, 2)
        )
    END AS yoy_growth
FROM cte
```

### Explanation
- **`CASE` logic**:
  - If `prev_spend` is `NULL` → no previous year to compare, so `yoy_growth` is `NULL`.  
  - Otherwise → calculate YoY growth.  
- **Formula**:
  \[
  \text{YoY Growth} = \frac{(\text{current_spend} - \text{prev_spend})}{\text{prev_spend}} \times 100
  \]
- **`COALESCE(prev_spend, 0)`** → Ensures that if `prev_spend` is missing, we substitute `0` (avoids errors).  
- **Multiple `CAST`s**:
  - Cast differences and divisions to `DECIMAL(10,2)` for precision.  
  - Final result is cast again to `DECIMAL(10,2)` so YoY growth is shown with two decimal places.

---

## Step 3: Final Output
The final result contains:
- `year` → Year of the transaction.  
- `product_id` → Product identifier.  
- `current_spend` → Spend in the current year.  
- `prev_spend` → Spend in the previous year (if available).  
- `yoy_growth` → Percentage growth compared to the previous year.  

---

## Key Insights
1. **`CAST`** is used multiple times to ensure financial precision (two decimal places).  
2. **`CASE`** handles missing previous years and avoids invalid growth calculations.  
3. **`LAG`** is the key window function that allows comparison with the previous year’s spend.  
4. The query is structured in two steps:
   - Build a base table (`cte`) with current and previous spend.  
   - Calculate YoY growth in the final `SELECT`.  

---
```
